# Capstone #2: Multi-Agent Research Team

*Notebook 16*

Put it all together.

Combine orchestration, parallel execution, built-in tools, and critique into a multi-agent pipeline.


---

## 🔧 Setup

In [ ]:
from pathlib import Path
from dotenv import load_dotenv
load_dotenv(dotenv_path=Path("..") / ".env")

from agents import Agent, CodeInterpreterTool, Runner, ToolCallItem, WebSearchTool

# Notebook-specific imports
import asyncio
import time

MODEL = "gpt-5-mini"
REASONING_MODEL = "gpt-5"


def called_tool_types(result) -> set[str]:
    """The set of hosted tool-call types a run invoked."""
    return {
        getattr(item.raw_item, "type", "")
        for item in result.new_items
        if isinstance(item, ToolCallItem)
    }


print("✅ Setup complete")

---

## 🏗️ System Architecture

This capstone combines Week 2 built-in tools with Week 3 orchestration:

- **Procedural orchestration**: Python drives the phases with `asyncio` and sequential `Runner.run()`

- **Lesson 14 (Parallel)**: researcher and analyst run concurrently

- **Lesson 15 (Debate & Critique)**: writer drafts, critic reviews, writer revises

We coordinate in code because the pipeline shape is fixed.

Use handoffs or agents-as-tools when the model should choose the next specialist.

**Pipeline:**
```
Research Question
        ↓
Pipeline coordinates specialists
   ↓               ↓
Researcher    Analyst      ← parallel execution
(web search)  (web + code)
   ↓               ↓
      Writer  ← receives both outputs
        ↓
      Critic  ← reviews and flags issues
        ↓
   Writer revision  ← addresses the critique
        ↓
   Final Report
```

**Reliability note:** The pipeline uses `asyncio.gather(return_exceptions=True)` for parallel steps.

If one specialist fails, the pipeline continues with labeled fallback text.

If both fail, it stops instead of drafting with no evidence.

⚠️ **Security note:** Treat web search results as untrusted input, potentially adversarial (more in Lesson 21).

---

## 🤖 Part 1: Build the Specialist Agents

In [ ]:
# Researcher: web search for current information
researcher_instructions = (
    "You are a research specialist. Search the web to find:\n"
    "- Current trends and recent developments\n"
    "- Key statistics and data points\n"
    "- Expert opinions and notable examples\n"
    "Summarize findings clearly with source attribution."
)

researcher_agent = Agent(
    name="Researcher",
    instructions=researcher_instructions,
    model=MODEL,
    tools=[WebSearchTool()]
)

# Analyst: web search for data, code interpreter for computation
analyst_instructions = (
    "You are a data analyst. For the given question:\n"
    "- Search for current numerical data with source attribution\n"
    "- Use code to compute derived metrics and identify patterns\n"
    "- Produce clear numerical summaries and show your calculations\n"
    "Never invent missing data."
)

# "auto" lets OpenAI manage the Code Interpreter container lifecycle
analyst_agent = Agent(
    name="Analyst",
    instructions=analyst_instructions,
    model=MODEL,
    tools=[
        WebSearchTool(),
        CodeInterpreterTool(tool_config={
            "type": "code_interpreter",
            "container": {"type": "auto"}
        })
    ]
)

# Writer: produces structured report from research
writer_instructions = (
    "You are a research report writer. Given research findings and analysis:\n"
    "- Write a structured report with clear sections\n"
    "- Use specific data points and cite sources\n"
    "- Keep language clear and professional\n"
    "- State missing evidence clearly. Never invent unavailable research or analysis.\n"
    "Format: Executive Summary, Key Findings, Data Analysis, Conclusion"
)

writer_agent = Agent(
    name="Writer",
    instructions=writer_instructions,
    model=MODEL
)

# Critic: reviews and challenges the report
# The Critic uses REASONING_MODEL because critique is judgment-heavy.
# The other specialists do well-scoped work on MODEL.
critic_instructions = (
    "You are a critical editor for research reports. Review the report and identify:\n"
    "- Unsupported claims or missing evidence\n"
    "- Logical gaps or weak conclusions\n"
    "- Missing context or important omissions\n"
    "Be direct and specific. Do not rewrite, only critique."
)

critic_agent = Agent(
    name="Critic",
    instructions=critic_instructions,
    model=REASONING_MODEL
)

# --------------------------------------------------------------
print("✅ Specialist agents ready")
print("    Researcher (web search)")
print("    Analyst (web search + code interpreter)")
print("    Writer")
print("    Critic")

## 🚀 Part 2: Define the Pipeline

We'll orchestrate the pipeline directly in code: parallel research, sequential writing and critique.

This gives us full visibility into each step.

This function is the reusable pattern.

Run independent specialists in parallel, and convert any failure into fallback text.

Then feed their outputs downstream one phase at a time.

In [ ]:
async def research_pipeline(question):
    """Full multi-agent research pipeline."""

    print(f"🔬 Research Question: {question}")
    print("=" * 60)
    start = time.time()

    # -------------------------------------------------------
    # Phase 1: Parallel research and analysis
    # -------------------------------------------------------
    print("\nPhase 1: Researching and analyzing in parallel...")

    research_result, analysis_result = await asyncio.gather(
        Runner.run(researcher_agent, input=question),
        Runner.run(
            analyst_agent,
            input=f"Analyze the quantitative aspects of this question: {question}"
        ),
        return_exceptions=True
    )

    # Handle partial failures: pipeline continues even if one specialist fails
    if isinstance(research_result, Exception):
        print(f"  ⚠️  Researcher failed: {research_result}")
        research_findings = "Web research unavailable for this run."
    else:
        research_findings = research_result.final_output

    if isinstance(analysis_result, Exception):
        print(f"  ⚠️  Analyst failed: {analysis_result}")
        analysis_findings = "Quantitative analysis unavailable for this run."
    else:
        analysis_findings = analysis_result.final_output

    # If BOTH specialists failed there is no evidence to write from: stop
    if isinstance(research_result, Exception) and isinstance(analysis_result, Exception):
        raise RuntimeError("Both specialists failed. No evidence is available for a report.")

    # Prove the hosted tools ran (empty set if a specialist failed)
    researcher_tools = (
        called_tool_types(research_result)
        if not isinstance(research_result, Exception)
        else set()
    )
    analyst_tools = (
        called_tool_types(analysis_result)
        if not isinstance(analysis_result, Exception)
        else set()
    )
    print(f"  Researcher web search: {'yes' if 'web_search_call' in researcher_tools else 'no'}")
    print(f"  Analyst web search: {'yes' if 'web_search_call' in analyst_tools else 'no'}")
    print(f"  Analyst code interpreter: {'yes' if 'code_interpreter_call' in analyst_tools else 'no'}")

    print(f"  ✓ Research phase complete ({time.time() - start:.1f}s)")

    # -------------------------------------------------------
    # Phase 2: Write the report
    # -------------------------------------------------------
    print("\nPhase 2: Writing report...")

    writer_input = (
        f"Write a research report on: {question}\n\n"
        f"RESEARCH FINDINGS:\n{research_findings}\n\n"
        f"DATA ANALYSIS:\n{analysis_findings}"
    )

    write_result = await Runner.run(writer_agent, input=writer_input)

    draft = write_result.final_output
    print(f"  ✓ Draft complete ({time.time() - start:.1f}s)")

    # -------------------------------------------------------
    # Phase 3: Critique the draft
    # -------------------------------------------------------
    print("\nPhase 3: Reviewing draft...")

    critique_input = (
        f"Use the supplied evidence to critique this report.\n\n"
        f"RESEARCH FINDINGS:\n{research_findings}\n\n"
        f"DATA ANALYSIS:\n{analysis_findings}\n\n"
        f"REPORT:\n{draft}"
    )

    critique_result = await Runner.run(critic_agent, input=critique_input)

    critique = critique_result.final_output
    print(f"  ✓ Critique complete ({time.time() - start:.1f}s)")

    # -------------------------------------------------------
    # Phase 4: Revise the report
    # -------------------------------------------------------
    print("\nPhase 4: Revising report...")

    revision_input = (
        f"Revise this report using the source evidence and critique.\n\n"
        f"RESEARCH FINDINGS:\n{research_findings}\n\n"
        f"DATA ANALYSIS:\n{analysis_findings}\n\n"
        f"ORIGINAL REPORT:\n{draft}\n\n"
        f"CRITIQUE:\n{critique}"
    )

    revision_result = await Runner.run(writer_agent, input=revision_input)

    final_report = revision_result.final_output

    total_time = time.time() - start
    print(f"  ✓ Revision complete ({total_time:.1f}s total)")

    return {
        "question": question,
        "research": research_findings,
        "analysis": analysis_findings,
        "draft": draft,
        "critique": critique,
        "final_report": final_report,
        "researcher_tools": sorted(researcher_tools),
        "analyst_tools": sorted(analyst_tools),
        "total_time": total_time
    }


# --------------------------------------------------------------
print("✅ research_pipeline() ready")

## 🎬 Part 3: Run the Pipeline

⚠️ **Cost note:** This demo makes five agent runs plus hosted web-search and Code Interpreter calls.

Cost and runtime vary with tool behavior and container lifecycle.

Budget a few minutes and some API credit per run.

In [ ]:
result = await research_pipeline(
    "What is the current state of electric vehicle adoption globally?"
)
print("\n" + "="*60)
print("FINAL REPORT")
print("="*60)
print(result["final_report"])

To watch the fallback text appear, temporarily break one specialist.

Set the researcher's `model` to a nonexistent name like `"gpt-nonexistent"` and re-run.

The failure surfaces inside `gather()`, and the fallback text takes its place.

The pipeline keeps running.

Restore the model name afterward.

---

## 💪 Practice Exercises

### Exercise 1: Add a Fact Checker

*Covers: pipeline extension, adding a verification phase*

Extend the pipeline with a `FactChecker` agent.

It checks the key claims in the final report against web sources before delivery.

This is one way you'll customize a pipeline in your own project.

Keep the orchestration shape, then insert or swap specialist phases where your workflow needs them.

In [ ]:
# --------------------------------------------------------------
# 💪 Exercise: Add a Fact Checker to the Pipeline
# --------------------------------------------------------------
# Objective: Add a verification step after the revision is complete.

async def research_pipeline_with_fact_check(question):
    """Research pipeline with fact checking step."""

    print(f"🔬 Research Question: {question}")
    print("=" * 60)
    start = time.time()

    # -------------------------------------------------------
    # Phase 1: Parallel research and analysis (provided)
    # -------------------------------------------------------
    print("\nPhase 1: Researching and analyzing in parallel...")

    results = await asyncio.gather(
        Runner.run(researcher_agent, input=question),
        Runner.run(
            analyst_agent,
            input=f"Analyze the quantitative aspects of this question: {question}"
        ),
        return_exceptions=True
    )

    # If a specialist failed, use fallback text so downstream phases still run
    research_result, analysis_result = results
    if isinstance(research_result, Exception):
        print(f"  ⚠️  Researcher failed: {research_result}")
        research_findings = "Web research unavailable for this run."
    else:
        research_findings = research_result.final_output

    if isinstance(analysis_result, Exception):
        print(f"  ⚠️  Analyst failed: {analysis_result}")
        analysis_findings = "Quantitative analysis unavailable for this run."
    else:
        analysis_findings = analysis_result.final_output

    # If BOTH specialists failed there is no evidence to write from: stop
    if isinstance(research_result, Exception) and isinstance(analysis_result, Exception):
        raise RuntimeError("Both specialists failed. No evidence is available for a report.")

    print(f"  ✓ Research complete ({time.time() - start:.1f}s)")

    # -------------------------------------------------------
    # Phase 2: Write the report (provided)
    # -------------------------------------------------------
    print("\nPhase 2: Writing report...")

    writer_input = (
        f"Write a research report on: {question}\n\n"
        f"RESEARCH FINDINGS:\n{research_findings}\n\n"
        f"DATA ANALYSIS:\n{analysis_findings}"
    )

    write_result = await Runner.run(writer_agent, input=writer_input)

    draft = write_result.final_output
    print(f"  ✓ Draft complete ({time.time() - start:.1f}s)")

    # -------------------------------------------------------
    # Phase 3: Critique (provided)
    # -------------------------------------------------------
    print("\nPhase 3: Reviewing draft...")

    critique_input = (
        f"Use the supplied evidence to critique this report.\n\n"
        f"RESEARCH FINDINGS:\n{research_findings}\n\n"
        f"DATA ANALYSIS:\n{analysis_findings}\n\n"
        f"REPORT:\n{draft}"
    )

    critique_result = await Runner.run(critic_agent, input=critique_input)

    critique = critique_result.final_output
    print(f"  ✓ Critique complete ({time.time() - start:.1f}s)")

    # -------------------------------------------------------
    # Phase 4: Revise (provided)
    # -------------------------------------------------------
    print("\nPhase 4: Revising report...")

    revision_input = (
        f"Revise this report using the source evidence and critique.\n\n"
        f"RESEARCH FINDINGS:\n{research_findings}\n\n"
        f"DATA ANALYSIS:\n{analysis_findings}\n\n"
        f"ORIGINAL REPORT:\n{draft}\n\n"
        f"CRITIQUE:\n{critique}"
    )

    revision_result = await Runner.run(writer_agent, input=revision_input)

    revised_report = revision_result.final_output
    print(f"  ✓ Revision complete ({time.time() - start:.1f}s)")

    # -------------------------------------------------------
    # Phase 5 (NEW): Fact check: YOUR CODE HERE
    # -------------------------------------------------------
    print("\nPhase 5: Fact checking...")

    # TODO 1: Create a fact_checker_agent with WebSearchTool
    #          Instructions: identify the 3 most important factual claims
    #          in the report, search to verify each one, and report
    #          which claims are confirmed, unverified, or questionable

    # TODO 2: Run the fact checker with revised_report as input

    # TODO 3: Store the result in fact_check_result

    # TODO 4: Confirm the fact checker actually searched:
    #          fact_checker_tools = called_tool_types(fact_check_result)
    #          and print whether "web_search_call" occurred

    # TODO 5: Print the fact check findings and the total elapsed time

    # TODO 6: Return a dict with question, final_report,
    #          fact_check=fact_check_result.final_output,
    #          fact_checker_tools=sorted(fact_checker_tools), and total_time
    return {}  # TODO: Replace with your dict


# --- Write your code below this line ---

# --------------------------------------------------------------
# TODO: Test your extended pipeline
# result = await research_pipeline_with_fact_check(
#      "What is the current state of electric vehicle adoption globally?"
# )

print("💡 Implement Phase 5 above, then run the pipeline!")

### Exercise 2: Evaluate the Pipeline Output

*Covers: judge agent pattern, scoring multi-agent pipeline output*

Apply the judge agent pattern from Lesson 07 to score the final report produced by the pipeline.

In [ ]:
# --------------------------------------------------------------
# 💪 Exercise 2: Evaluate the Pipeline Output
# --------------------------------------------------------------
# Objective: Apply the Lesson 07 judge agent pattern to multi-agent output.
# Note: run the demo cell above first to populate result.

from pydantic import BaseModel, Field
from typing import Annotated


# TODO 1: Define a ReportEval Pydantic model with:
#           - score: Annotated[int, Field(ge=1, le=5)]
#           - reasoning: str
#          (Do not make `passed` a model field: derive it in Python below.)

# TODO 2: Create a judge agent using REASONING_MODEL with output_type=ReportEval
#           Instructions: score clarity, evidence quality, and logical structure
#           (1-5), using the supplied source evidence

# TODO 3: Judge the captured output, without rerunning the pipeline. Build the
#           judge input from the pipeline result:
#             f"SOURCE RESEARCH:\n{result['research']}\n\n"
#             f"SOURCE ANALYSIS:\n{result['analysis']}\n\n"
#             f"FINAL REPORT:\n{result['final_report']}"

# TODO 4: Compute passed = evaluation.score >= 3 in Python
#           Print the score, reasoning, and pass/fail verdict

# TODO 5: Make it repeatable: define a fixed sample report AND a fixed evidence
#           packet in strings, run the judge on them, and compare the score to an
#           expected minimum (a mini golden test you can rerun after changes)

# --- Write your code below this line ---

---

## 🎯 Key Takeaways

**Multi-agent pipelines combine patterns:**

- Parallel calls can reduce Phase 1 wall time when they overlap

- Sequential phases run when each depends on the previous output

- A single critique pass creates a deliberate revision step
<br>
<br>

**Use models by role, and pass evidence forward:**

- This demo uses `REASONING_MODEL` for judgment-heavy critique and `MODEL` for narrower specialist tasks

- A new `Runner.run()` does not inherit an earlier run's input

- Pass source evidence again when critique or revision needs it
<br>
<br>

**Prove what actually happened:**

- Inspect `new_items` to prove hosted tools ran

- Each phase's output becomes the next phase's input, coordinated in plain code

- Because it is plain code, phases are easy to add, remove, or reorder
<br>
<br>

**Parallel phases need failure handling:**

- `return_exceptions=True` returns failures as values to inspect

- One failed source gets visible fallback text

- If every evidence source fails, stop before drafting

---

## 📍 Next Step

**Notebook 17: Sessions & Conversation State**  

Give your agents conversation memory so they remember prior turns across separate runs.

---

##### 🔧 [Troubleshooting Guide](https://github.com/barrettscott/openai-agents/blob/main/TROUBLESHOOTING.md#lesson-16-capstone-2--multi-agent-research-team)

---